# Treasury yield-curve PCA and portfolio risk

Maintainer: **Somya Harjai** (`sharjai-code`). An independently coded Python adaptation inspired by Andrea Ranzato's [dissertation repository](https://github.com/andreranza/pca-yield-curve-modelling). Development was assisted by ChatGPT.

This notebook uses a frozen 2025 Federal Reserve GSW zero-coupon dataset. It studies decomposition of observed yield changes and static rate risk; it does not estimate corporate-credit risk or forecast trading returns.

Run cells in order. The first cell finds the repository root whether Jupyter starts in the root or the notebooks folder.

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))
from yield_curve_pca.data import load_yields, yield_changes_bp
from yield_curve_pca.pca import fit_pca, block_bootstrap, rolling_pca
from yield_curve_pca.curves import Bond, price_portfolio, key_rate_dv01

yields = load_yields(ROOT / 'data/gsw_2025.csv')
changes = yield_changes_bp(yields)
print(f'{len(yields)} complete curves; {len(changes)} daily changes')
print(yields.iloc[[0, -1]].round(4).to_string())

249 complete curves; 248 daily changes
maturity_years      1       2       3       5       7       10      20      30
Date                                                                          
2025-01-02      4.1919  4.2189  4.2541  4.3406  4.4381  4.5845  4.8934  4.8474
2025-12-31      3.5082  3.4717  3.5167  3.7092  3.9374  4.2568  4.9147  5.1025


## 1. Fit PCA in two input spaces

For centered daily changes $X$ in basis points, covariance PCA uses $X=USV^T$. Eigenvalues equal $s_j^2/(n-1)$. Correlation PCA first divides each maturity by its training sample standard deviation. The two explained-variance percentages have different denominators.

Signs are arbitrary. Factor labels such as level, slope, and curvature require inspection of the actual coefficients.

In [2]:
covariance = fit_pca(changes, 'covariance')
correlation = fit_pca(changes, 'correlation')
for label, model in [('covariance', covariance), ('correlation', correlation)]:
    print(label, 'top-three variance:', f'{model.explained_variance_ratio[:3].sum():.4%}')
print(pd.DataFrame(covariance.components[:3].T, index=yields.columns, columns=['PC1','PC2','PC3']).round(4).to_string())

covariance top-three variance: 98.6756%
correlation top-three variance: 98.5531%
                   PC1     PC2     PC3
maturity_years                        
1               0.2279 -0.3275  0.3656
2               0.3530 -0.4032 -0.0248
3               0.3948 -0.3371 -0.0720
5               0.4082 -0.1332  0.0402
7               0.4031  0.0387  0.0694
10              0.3937  0.2109 -0.1113
20              0.3424  0.4455 -0.6346
30              0.2574  0.5953  0.6625


## 2. Chronological holdout reconstruction

The final 20% of observed changes are held out. Means, scales, and loadings are frozen at the training cutoff. This tests reconstruction of observed shocks, not a forecast of those shocks. Errors are reported in raw basis points for a common comparison across PCA methods.

In [3]:
cutoff = int(len(changes) * 0.8)
train, test = changes.iloc[:cutoff], changes.iloc[cutoff:]
print('Training ends:', train.index[-1].date(), '| Holdout starts:', test.index[0].date())
for mode in ['covariance', 'correlation']:
    model = fit_pca(train, mode)
    for k in [1, 2, 3]:
        error = test.to_numpy() - model.reconstruct(test.to_numpy(), k)
        print(mode, k, 'PCs | holdout RMSE:', round(float(np.sqrt(np.mean(error**2))), 4), 'bp')

Training ends: 2025-10-17 | Holdout starts: 2025-10-20
covariance 1 PCs | holdout RMSE: 1.4101 bp
covariance 2 PCs | holdout RMSE: 0.7433 bp
covariance 3 PCs | holdout RMSE: 0.4006 bp
correlation 1 PCs | holdout RMSE: 1.4144 bp
correlation 2 PCs | holdout RMSE: 0.755 bp
correlation 3 PCs | holdout RMSE: 0.5149 bp


## 3. Block-bootstrap uncertainty

Circular moving blocks retain local temporal dependence. This run uses 500 repetitions, ten-observation blocks, and seed 42. Percentile intervals are sample-dependent and do not protect against nonstationarity. Loading intervals are pointwise; individual factors become difficult to identify near equal eigenvalues.

In [4]:
variance_interval, loading_intervals = block_bootstrap(changes, repetitions=500, block_length=10, seed=42)
print(variance_interval.round(6).to_string())
print(loading_intervals.head(8).round(4).to_string(index=False))

       estimate  lower_95  upper_95
top_3  0.986756  0.984393   0.99234
factor  maturity_years  estimate  lower_95  upper_95
   PC1             1.0    0.2279    0.1939    0.2687
   PC1             2.0    0.3530    0.3219    0.3862
   PC1             3.0    0.3948    0.3694    0.4182
   PC1             5.0    0.4082    0.3942    0.4201
   PC1             7.0    0.4031    0.3880    0.4148
   PC1            10.0    0.3937    0.3739    0.4099
   PC1            20.0    0.3424    0.3007    0.3797
   PC1            30.0    0.2574    0.2119    0.2960


## 4. Rolling stability

Each fit uses only a trailing 63-observation window. Factors are matched to the first window by absolute cosine similarity and then sign-aligned. The largest principal angle describes changes in the retained subspace and is invariant to sign and within-subspace rotations.

In [5]:
rolling = rolling_pca(changes, window=63)
print(rolling[['top3_variance','max_subspace_angle_degrees','PC1_cosine','PC2_cosine','PC3_cosine']].describe().round(4).to_string())

       top3_variance  max_subspace_angle_degrees  PC1_cosine  PC2_cosine  PC3_cosine
count       186.0000                    186.0000    186.0000    186.0000    186.0000
mean          0.9937                     66.0877      0.9959      0.9604      0.3212
std           0.0020                     27.9734      0.0033      0.0163      0.3299
min           0.9886                      0.0000      0.9851      0.9334      0.0014
25%           0.9925                     68.8765      0.9936      0.9449      0.1301
50%           0.9940                     78.1312      0.9972      0.9596      0.1906
75%           0.9953                     82.0173      0.9983      0.9680      0.3331
max           0.9965                     89.9055      1.0000      1.0000      1.0000


## 5. Portfolio valuation and sensitivities

The hypothetical portfolio has $1m face each of 2-, 5-, 10-, and 30-year bonds with 4% semiannual coupons. Valuation occurs on a coupon date with no accrued interest. GSW spot yields are continuously compounded: $D(t)=e^{-y(t)t}$. Log discount factors are interpolated between nodes with $D(0)=1$.

Positive DV01 is the dollar benefit of a one-bp yield fall, so first-order P&L is $-\mathrm{DV01}^{T}\Delta y_{bp}$.

In [6]:
maturities = yields.columns.to_numpy(dtype=float)
base_curve = yields.iloc[-1].to_numpy()
bonds = [Bond(m) for m in [2, 5, 10, 30]]
base_value = price_portfolio(bonds, maturities, base_curve)
dv01 = key_rate_dv01(bonds, maturities, base_curve)
print('Portfolio value ($):', round(base_value, 2))
print('Parallel DV01 ($/bp):', round(float(dv01.sum()), 2))
print(pd.Series(dv01, index=maturities, name='key_rate_dv01_usd_per_bp').round(2).to_string())

Portfolio value ($): 3858203.04
Parallel DV01 ($/bp): 2857.4
1.0      15.45
2.0     214.60
3.0      47.95
5.0     502.94
7.0     104.51
10.0    851.70
20.0    307.53
30.0    812.71


## 6. Full repricing versus linear risk

For a large shock, convexity makes the full repricing P&L differ from the linear estimate. PCA shocks use one sample standard deviation of each factor; their direction follows the chosen sign convention.

In [7]:
rows=[]
for shock_bp in [-100, 100]:
    shock=np.full(len(maturities), float(shock_bp))
    exact=price_portfolio(bonds,maturities,base_curve+shock/100)-base_value
    linear=float(-shock @ dv01)
    rows.append({'parallel_shock_bp':shock_bp,'full_repricing_usd':exact,'linear_usd':linear,'residual_usd':exact-linear})
print(pd.DataFrame(rows).round(2).to_string(index=False))
for j in range(3):
    shock=np.sqrt(covariance.eigenvalues[j])*covariance.basis_bp[j]
    exact=price_portfolio(bonds,maturities,base_curve+shock/100)-base_value
    print(f'PC{j+1} +1 sigma: full repricing P&L ${exact:,.2f}')

 parallel_shock_bp  full_repricing_usd  linear_usd  residual_usd
              -100           308549.77   285740.08      22809.69
               100          -266036.75  -285740.08      19703.33
PC1 +1 sigma: full repricing P&L $-12,483.56
PC2 +1 sigma: full repricing P&L $-3,344.68
PC3 +1 sigma: full repricing P&L $-459.06


## Generated figures

Run `python -m yield_curve_pca.analysis` from the repository root to rebuild these figures and all output tables.

![Research overview](../results/research_overview.png)

![Portfolio stress scenarios](../results/stress_scenarios.png)

## Interpretation

The 2025 sample supports a compact description of observed daily curve variation, but it cannot establish stability across long-run market regimes. The first three coefficients need not produce textbook shapes. Compare covariance and correlation methods by raw-bp errors, investigate longer samples and alternative block lengths, and distinguish static stress P&L from realized returns.

See the [research report](../results/research_report.md), [attribution](../NOTICE.md), and [contribution notes](../CONTRIBUTIONS.md).